# E6 · Mejoras del stacking: metamodelo y *seed bagging*
**Parte de:** E0. E3 mostró que agregar modelos base no ayuda, así que aquí se mantienen los 5 originales y solo se cambia **cómo se combinan**.

**Hipótesis.**
1. **Metamodelo:** otra regularización (`C`), sin `class_weight`, un metamodelo no lineal (LightGBM) o pasarle algunas variables de léxico clave podría combinar mejor.
2. ***Seed bagging*:** cada modelo base OOF se entrena con una partición al azar, así que sus probabilidades tienen varianza. Si se promedian los OOF de varias semillas antes del stacking, el metamodelo recibe probabilidades más estables. En el test, la contraparte es promediar los 5 × k modelos de partición, entrenados con 4/5 del train igual que los OOF.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **Metamodelo** | El modelo de segundo nivel del stacking |
| **Seed bagging** | Repetir el entrenamiento con varias semillas de partición y promediar las predicciones |
| **Varianza** | Cuánto cambian las predicciones si se entrena con otra muestra de los datos |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
B5 = tb.BASES_ORIGINALES
st = lambda **kw: tb.evaluar(d, {s: tb.stacking(d, B5, s, con_test=False, **kw)[0] for s in tb.SEEDS})
cols = [d.lex_nombres.index(c) for c in ['comillas', 'mencion_inicial', 'fem_sin_intensificador', 'insulto_grupo',
                                         'migracion', 'mujer_generica', 'hijo_de_puta', 'puta_intensificador']]
R = {'E0 · meta LR C=1, balanced': st()}
for C in [0.1, 0.3, 3.0]:
    R[f'meta LR C={C}'] = st(meta=lambda C=C: LogisticRegression(C=C, max_iter=3000, class_weight='balanced'))
R['meta LR sin class_weight'] = st(meta=lambda: LogisticRegression(C=1.0, max_iter=3000))
R['meta LightGBM (4 hojas)'] = st(meta=lambda: lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=4,
                                  min_child_samples=20, class_weight='balanced', verbose=-1, random_state=0))
R['meta + 8 variables de léxico'] = st(extra_tr=d.L_TR[:, cols], extra_te=d.L_TE[:, cols])
tb.tabla(R)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
"E0 · meta LR C=1, balanced",0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
meta LR C=0.1,0.8045,0.8371,0.7990,0.8135,0.0025,0.7766,0.817 / 0.812 / 0.811
meta LR C=0.3,0.8052,0.8374,0.7978,0.8135,0.0033,0.7762,0.818 / 0.812 / 0.811
meta LR C=3.0,0.8061,0.8312,0.7985,0.8120,0.0024,0.7782,0.815 / 0.809 / 0.812
meta LR sin class_weight,0.8024,0.8281,0.7900,0.8068,0.0015,0.7745,0.805 / 0.808 / 0.807
meta LightGBM (4 hojas),0.8080,0.8323,0.7938,0.8114,0.0024,0.7712,0.810 / 0.809 / 0.815
meta + 8 variables de léxico,0.8060,0.8324,0.7987,0.8124,0.0038,0.7766,0.818 / 0.810 / 0.810


**Lectura:** ninguna variante del metamodelo se aleja de E0 más que la desviación entre semillas (±0,002-0,004). `C=0,1-0,3` gana 0,0009, que es ruido. Quitar `class_weight` baja el F1 de la clase positiva, como se esperaba.

## 2. *Seed bagging*
### 2.1 ¿El promedio de semillas ayuda a cada modelo por separado?
Esta comparación no involucra al metamodelo: si el promedio mejora un modelo individual, el efecto es reducción de varianza y no una fuga del stacking.

In [3]:
filas = []
for b in B5:
    una = np.mean([tb.metricas(d.Y, tb.binarizar(tb.oof(d, b, s)))['F1 medio'] for s in tb.SEEDS])
    bag = tb.metricas(d.Y, tb.binarizar(np.mean([tb.oof(d, b, s) for s in tb.SEEDS], 0)))['F1 medio']
    filas.append({'modelo': b, 'F1 medio (1 semilla, promedio)': una, 'F1 medio (OOF promediado 3 semillas)': bag, 'diferencia': bag - una})
pd.DataFrame(filas).set_index('modelo').round(4)

,"F1 medio (1 semilla, promedio)",F1 medio (OOF promediado 3 semillas),diferencia
modelo,,,
CNB,0.7463,0.7513,0.0050
LR,0.8023,0.8040,0.0017
SVM,0.8017,0.8063,0.0046
NB-LR,0.8010,0.8052,0.0042
Powerset,0.8037,0.8057,0.0020


### 2.2 Stacking sobre los modelos base promediados

In [4]:
S5 = (42, 43, 44, 45, 46)
for s in S5:                      # OOF y predicciones de test por partición (en caché si ya existen)
    for b in B5:
        tb.oof(d, b, s); tb.pred_test_folds(d, b, s)
RB = {'E0 · sin bagging': R['E0 · meta LR C=1, balanced']}
for nombre, sb in [('bagging 3 semillas (42-44)', S5[:3]), ('bagging 5 semillas (42-46)', S5)]:
    RB[nombre] = tb.evaluar(d, {s: tb.stacking_bagged(d, B5, sb, s, con_test=False)[0] for s in tb.SEEDS})
tb.tabla(RB)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E0 · sin bagging,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
bagging 3 semillas (42-44),0.8123,0.8422,0.8029,0.8191,0.0014,0.7834,0.818 / 0.818 / 0.821
bagging 5 semillas (42-46),0.8092,0.8368,0.7988,0.8150,0.0007,0.7803,0.814 / 0.815 / 0.815


**Cuidado al leer esta tabla.** En las filas con bagging, la desviación ± solo refleja la semilla del **metamodelo**: los modelos base ya están promediados y son una sola realización. La diferencia entre 3 semillas (0,819) y 5 semillas (0,815) muestra que esa realización tiene su propio ruido, de unos ±0,004. La estimación más estable es la de 5 semillas: **+0,0024 sobre E0, y mejora en las 3 semillas del metamodelo**.

## 3. Submission

In [5]:
_, Pt = tb.stacking_bagged(d, B5, S5, 42)
_, Pt_full = tb.stacking(d, B5, 42)
print('Correlación de probabilidades de test (bagging vs. reentrenar con todo el train):',
      [round(np.corrcoef(Pt[:, j], Pt_full[:, j])[0, 1], 3) for j in range(3)])
sub = tb.guardar_submission(d, tb.binarizar(Pt), 'toxic_submission_E6_bagging5.csv')

Correlación de probabilidades de test (bagging vs. reentrenar con todo el train): [np.float64(0.999), np.float64(0.999), np.float64(0.999)]
[     4s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E6_bagging5.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.39), 'TR': np.float64(0.297), 'AG': np.float64(0.368)}
Cambios frente a la base 0,79160: 19 filas ({'HS': 8, 'TR': 7, 'AG': 10})


## Conclusión: **candidato a envío (mejora pequeña, riesgo bajo)**
- Las variantes del metamodelo no ayudan: E0 ya está bien regularizado.
- El *seed bagging* mejora cada modelo base por separado (+0,2 a +0,5 puntos) y el stacking en +0,0024 con 5 semillas. Es un poco menos que el umbral de 0,003, pero es consistente en las 3 semillas y tiene una explicación clara (reducción de varianza).
- En el test cambia **solo 19 de 1.154 filas** frente a la submission de 0,79160 (correlación de probabilidades 0,999). Es una versión **más estable** del mismo modelo, no un modelo distinto, y por eso tiene poco riesgo.
- Archivo: `submissions/toxic_submission_E6_bagging5.csv`.